# Predictive Modeling Using Machine Learning — Customer Churn

Predict whether a customer will churn using Logistic Regression, a Decision Tree, and a Random Forest, then compare them with confusion matrices, ROC curves, and standard classification metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score, classification_report
)

RANDOM_STATE = 42

## 1. Load the data

In [ ]:
df = pd.read_csv("../data/customer_churn.csv")
print(df.shape)
df.head()

In [ ]:
df["churn"].value_counts(normalize=True)

## 2. Prepare features and target

In [ ]:
target_col = "churn"
id_col = "customer_id"

X = df.drop(columns=[target_col, id_col])
y = (df[target_col] == "Yes").astype(int)

numeric_features = [
    "age", "senior_citizen", "tenure_months", "monthly_charges",
    "total_charges", "num_support_calls",
]
categorical_features = [c for c in X.columns if c not in numeric_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
len(X_train), len(X_test)

## 3. Build the preprocessing + modeling pipelines

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1),
}

## 4. Train and evaluate each model

In [ ]:
results = {}
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "y_pred": y_pred,
        "y_prob": y_prob,
    }
    print(f"{name}: accuracy={results[name]['accuracy']:.3f}, "
          f"f1={results[name]['f1']:.3f}, roc_auc={results[name]['roc_auc']:.3f}")

## 5. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res["y_pred"])
    ax.imshow(cm, cmap="Blues")
    ax.set_title(f"{name}\nConfusion Matrix")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["No Churn", "Churn"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["No Churn", "Churn"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.show()

## 6. ROC curves

In [ ]:
plt.figure(figsize=(6.5, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['roc_auc']:.3f})", linewidth=2)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Model Comparison")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 7. Feature importance (Random Forest)

In [ ]:
rf_pipe = fitted_pipelines["Random Forest"]
feature_names = rf_pipe.named_steps["preprocess"].get_feature_names_out()
importances = rf_pipe.named_steps["model"].feature_importances_
imp_series = pd.Series(importances, index=feature_names).sort_values(ascending=True).tail(15)

plt.figure(figsize=(8, 6))
imp_series.plot(kind="barh", color="#4C72B0")
plt.title("Top 15 Feature Importances — Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 8. Full classification reports

In [ ]:
for name, res in results.items():
    print(f"\n--- {name} ---")
    print(classification_report(y_test, res["y_pred"], target_names=["No Churn", "Churn"]))